# Análisis de Eficiencia Computacional: Pandas vs Polars
### Aplicado a Telemetría de Centros de Datos en la Nube

**Materia:** Fundamentos de Arquitectura de Computadoras, Sistemas Operativos y Redes
**Tema:** Almacenamiento en la Nube: Centros de Datos y su Impacto
**Anexo del Informe Final Extendido**

---

## Contexto del problema

Este notebook reproduce y documenta el benchmarking comparativo entre **Pandas** (evaluación *Eager*) y **Polars** (evaluación *Lazy*) al procesar un dataset sintético de **15 millones de registros** que simula telemetría real de un centro de datos: identificador de datacenter, país de origen, latencia en milisegundos y bytes transferidos.

El objetivo es medir tres métricas de rendimiento:
- **Tiempo de ejecución** (segundos)
- **Pico de uso de memoria RAM** (MB)
- **Utilización de CPU** (%)

y vincular los resultados con conceptos teóricos de la materia: **localidad espacial**, **líneas de caché**, **el GIL de Python**, **evaluación Lazy vs Eager** y **paralelismo multi-core**.

## Paso 1 — Instalación e importación de librerías

Si es la primera vez que ejecutás este notebook, la siguiente celda instala todas las dependencias necesarias. Si ya las tenés instaladas, la podés correr igual sin problema (no reinstala nada de más).

In [ ]:
# Instalación de librerías necesarias (correr una sola vez)
%pip install pandas polars numpy psutil matplotlib --quiet
print("Librerías instaladas correctamente.")
print("(Si ves un aviso de 'externally-managed-environment', es normal en algunos")
print("sistemas Linux/Mac y no afecta el resto del notebook)")

In [ ]:
import pandas as pd
import polars as pl
import time
import tracemalloc
import psutil
import threading
import os
import numpy as np
import matplotlib.pyplot as plt

# Carpeta donde se guardarán los gráficos para insertar en el informe
CARPETA_GRAFICOS = "graficos_resultado"
os.makedirs(CARPETA_GRAFICOS, exist_ok=True)

print("Librerías importadas correctamente.")
print(f"Pandas version: {pd.__version__}")
print(f"Polars version: {pl.__version__}")

## Paso 2 — Definición del pipeline de operaciones

El experimento consiste en un pipeline de **3 operaciones encadenadas**, típicas en el análisis de logs de un datacenter:

1. **Lectura de disco**: cargar el archivo CSV con 15 millones de filas.
2. **Filtrado**: quedarnos solo con los registros de datacenters ubicados en Argentina (`pais_origen == 'AR'`).
3. **Agrupación y agregación**: calcular el promedio de latencia (`latencia_ms`) agrupado por `id_datacenter`.

Este pipeline representa un caso real: un ingeniero de datos queriendo saber **qué datacenters argentinos tienen mejor o peor latencia promedio**, a partir de millones de registros de telemetría.

In [ ]:
ARCHIVO = 'dataset_telemetria_cloud_15M.csv'

def generar_datos_cloud(filas=15_000_000):
    """
    Genera un dataset sintético que simula telemetría real de un datacenter:
    - id_datacenter: identificador del servidor físico
    - pais_origen: país donde se originó la petición
    - latencia_ms: latencia de red registrada (distribución normal ~120ms, similar a la
      latencia real Argentina-EE.UU. documentada en la sección 2.3 del informe)
    - bytes_transferidos: tamaño del paquete de datos transferido
    """
    if not os.path.exists(ARCHIVO):
        print(f"Generando dataset sintético de {filas:,} filas... (puede tardar uno o dos minutos)")
        df = pd.DataFrame({
            'id_datacenter': np.random.randint(100, 999, filas),
            'pais_origen': np.random.choice(['AR', 'BR', 'US', 'CL', 'UY'], filas),
            'latencia_ms': np.random.normal(120, 30, filas),
            'bytes_transferidos': np.random.randint(500, 5_000_000, filas)
        })
        df.to_csv(ARCHIVO, index=False)
        print("Dataset generado y guardado correctamente.")
    else:
        print("El dataset ya existe, se reutiliza el archivo existente.")

generar_datos_cloud()

# Verificación rápida del tamaño del archivo en disco
tamano_mb = os.path.getsize(ARCHIVO) / (1024**2)
print(f"Tamaño del archivo en disco: {tamano_mb:.2f} MB")

## Función auxiliar de medición de CPU

Para medir la utilización de CPU mientras corre el pipeline, usamos un *thread* paralelo que va tomando muestras con `psutil.cpu_percent()` cada 0.1 segundos, sin interferir con la medición principal de tiempo y memoria.

In [ ]:
def medir_cpu(stop_event, cpu_uso):
    """Hilo en segundo plano que muestrea el % de uso de CPU cada 0.1s"""
    while not stop_event.is_set():
        cpu_uso.append(psutil.cpu_percent(interval=0.1))

## Paso 3 — Benchmarking con Pandas (Evaluación *Eager*)

Pandas ejecuta cada operación del pipeline de forma **inmediata** (Eager): apenas se llama a `read_csv()`, carga el archivo completo en RAM. Luego cada filtro y `groupby()` genera una **copia intermedia** en memoria.

Además, Pandas opera en formato **row-major** (por filas), lo que **rompe la localidad espacial** al procesar columnas individuales — el procesador debe saltar entre posiciones de memoria no contiguas.

Por último, Pandas está limitado por el **GIL (Global Interpreter Lock)** de Python: aunque la máquina tenga múltiples núcleos, solo uno puede ejecutar bytecode Python a la vez.

In [ ]:
def test_pandas():
    print("Iniciando análisis con Pandas (Eager Execution)...")
    tracemalloc.start()
    start_time = time.time()

    stop_event = threading.Event()
    cpu_uso = []
    t = threading.Thread(target=medir_cpu, args=(stop_event, cpu_uso))
    t.start()

    # ---- Pipeline secuencial, limitado por el GIL ----
    df = pd.read_csv(ARCHIVO)
    df_filtrado = df[df['pais_origen'] == 'AR']
    resultado = df_filtrado.groupby('id_datacenter')['latencia_ms'].mean()
    # ---------------------------------------------------

    stop_event.set()
    t.join()

    tiempo_total = time.time() - start_time
    memoria_mb = tracemalloc.get_traced_memory()[1] / 10**6
    tracemalloc.stop()
    cpu_promedio = sum(cpu_uso) / len(cpu_uso) if cpu_uso else 0

    print(f"[Pandas] Tiempo: {tiempo_total:.2f} s | RAM: {memoria_mb:.2f} MB | CPU: {cpu_promedio:.2f}%")
    return tiempo_total, memoria_mb, cpu_promedio, resultado

tiempo_pandas, ram_pandas, cpu_pandas, resultado_pandas = test_pandas()

## Paso 4 — Benchmarking con Polars (Evaluación *Lazy*)

Polars usa **evaluación diferida (Lazy)**: con `scan_csv()` no se carga el archivo a memoria, se construye un **plan de consultas**. Recién al llamar `.collect()` el planificador interno **fusiona las operaciones** (filtro + agrupación) y lee del disco solo las columnas necesarias.

Internamente usa el formato columnar **Apache Arrow**, que coloca los datos del mismo tipo de forma **contigua en memoria**, maximizando la localidad espacial y llenando eficientemente las líneas de caché L1/L2 del procesador (habilitando además vectorización **SIMD**).

Al estar escrito en **Rust**, Polars evade el GIL de Python y distribuye el trabajo en **paralelismo real multi-core**.

In [ ]:
def test_polars():
    print("Iniciando análisis con Polars (Lazy Evaluation & Apache Arrow)...")
    tracemalloc.start()
    start_time = time.time()

    stop_event = threading.Event()
    cpu_uso = []
    t = threading.Thread(target=medir_cpu, args=(stop_event, cpu_uso))
    t.start()

    # ---- Pipeline Lazy: se planifica y optimiza antes de ejecutar ----
    resultado = (
        pl.scan_csv(ARCHIVO)
        .filter(pl.col('pais_origen') == 'AR')
        .group_by('id_datacenter')
        .agg(pl.col('latencia_ms').mean())
        .collect()
    )
    # --------------------------------------------------------------------

    stop_event.set()
    t.join()

    tiempo_total = time.time() - start_time
    memoria_mb = tracemalloc.get_traced_memory()[1] / 10**6
    tracemalloc.stop()
    cpu_promedio = sum(cpu_uso) / len(cpu_uso) if cpu_uso else 0

    print(f"[Polars] Tiempo: {tiempo_total:.2f} s | RAM: {memoria_mb:.2f} MB | CPU: {cpu_promedio:.2f}%")
    return tiempo_total, memoria_mb, cpu_promedio, resultado

tiempo_polars, ram_polars, cpu_polars, resultado_polars = test_polars()

## Paso 5 — Tabla resumen y visualización comparativa

Consolidamos los resultados de ambas librerías en una tabla, y generamos 3 gráficos de barras (uno por métrica) para insertar directamente en el informe final.

In [ ]:
tabla_resultados = pd.DataFrame({
    'Métrica': ['Tiempo de ejecución (s)', 'Pico de RAM (MB)', 'CPU promedio (%)'],
    'Pandas (Eager)': [round(tiempo_pandas, 2), round(ram_pandas, 2), round(cpu_pandas, 2)],
    'Polars (Lazy)': [round(tiempo_polars, 2), round(ram_polars, 2), round(cpu_polars, 2)]
})
tabla_resultados['Mejora (x veces)'] = (tabla_resultados['Pandas (Eager)'] / tabla_resultados['Polars (Lazy)']).round(2)
tabla_resultados

In [ ]:
# Gráfico 1: Tiempo de ejecución
fig, ax = plt.subplots(figsize=(6, 4.5))
librerias = ['Pandas\n(Eager)', 'Polars\n(Lazy)']
tiempos = [tiempo_pandas, tiempo_polars]
colores = ['#E57373', '#64B5F6']
barras = ax.bar(librerias, tiempos, color=colores, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Segundos')
ax.set_title('Tiempo de Ejecución: Pandas vs Polars')
for barra, valor in zip(barras, tiempos):
    ax.text(barra.get_x() + barra.get_width()/2, valor, f'{valor:.2f}s',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CARPETA_GRAFICOS}/01_tiempo_ejecucion.png', dpi=150)
plt.show()

In [ ]:
# Gráfico 2: Pico de memoria RAM
fig, ax = plt.subplots(figsize=(6, 4.5))
rams = [ram_pandas, ram_polars]
barras = ax.bar(librerias, rams, color=colores, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Megabytes (MB)')
ax.set_title('Pico de Consumo de Memoria RAM: Pandas vs Polars')
for barra, valor in zip(barras, rams):
    ax.text(barra.get_x() + barra.get_width()/2, valor, f'{valor:.1f} MB',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CARPETA_GRAFICOS}/02_memoria_ram.png', dpi=150)
plt.show()

In [ ]:
# Gráfico 3: Utilización promedio de CPU
fig, ax = plt.subplots(figsize=(6, 4.5))
cpus = [cpu_pandas, cpu_polars]
barras = ax.bar(librerias, cpus, color=colores, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Porcentaje (%)')
ax.set_ylim(0, 100)
ax.set_title('Utilización Promedio de CPU: Pandas vs Polars')
for barra, valor in zip(barras, cpus):
    ax.text(barra.get_x() + barra.get_width()/2, valor, f'{valor:.1f}%',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CARPETA_GRAFICOS}/03_uso_cpu.png', dpi=150)
plt.show()

print(f"\nLos 3 gráficos se guardaron en la carpeta '{CARPETA_GRAFICOS}/' listos para insertar en el informe.")

## Conclusión y diagnóstico del cuello de botella

A partir de los resultados obtenidos, podemos diagnosticar el tipo de cuello de botella que enfrenta cada librería, conectando con los conceptos teóricos de la materia:

- **Pandas (CPU-bound por el GIL)**: la utilización de CPU se mantiene baja en términos relativos al total de núcleos disponibles porque el GIL obliga a procesar todo en un solo núcleo de forma secuencial. El cuello de botella aquí es **computacional**: el procesador está limitado artificialmente por el lenguaje, no por el hardware.

- **Polars (paralelismo real + localidad espacial)**: al evadir el GIL (por estar escrito en Rust) y aprovechar el formato columnar Apache Arrow, Polars reduce drásticamente tanto el tiempo de ejecución como el pico de memoria. El menor consumo de RAM se explica porque la evaluación *Lazy* nunca llega a materializar el dataset completo en memoria, solo los chunks necesarios para resolver la consulta.

**Relación con la arquitectura de un datacenter real**: este mismo principio se aplica a escala macroscópica en los centros de datos que estudiamos en el informe, la eficiencia del software que corre sobre el hardware (CPUs EPYC/Xeon, RAM ECC, NVMe) determina si esa infraestructura física se aprovecha al máximo o se desperdicia, con impacto directo en el consumo energético y, por extensión, en la huella de carbono del datacenter (sección 4.1 del informe).